# Predictive Analysis — Falcon 9 Landing Classification
Compares Logistic Regression, Support Vector Machine, Decision Tree and K-Nearest Neighbors. This notebook calculates test metrics from the actual predictions. It uses a stratified split and a pipeline so scaling is learned from training data only.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

X_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv"
Y_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
X = pd.read_csv(X_URL)
y_df = pd.read_csv(Y_URL)
if "Class" not in y_df.columns:
    raise KeyError(f"Expected target column 'Class'; found {y_df.columns.tolist()}")
y = y_df["Class"].astype(int)
print("X shape:", X.shape, "y shape:", y.shape)
print("Target distribution:")
display(y.value_counts().to_frame("count"))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=2, stratify=y
)
min_class_count = int(y_train.value_counts().min())
cv_splits = min(5, min_class_count)
if cv_splits < 2:
    raise ValueError("Not enough training examples per class for cross-validation.")
cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=2)
print("Train:", X_train.shape, "Test:", X_test.shape, "CV folds:", cv_splits)

In [ ]:
models = {
    "Logistic Regression": (
        Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000))]),
        {"model__C": [0.01, 0.1, 1, 10]}
    ),
    "SVM": (
        Pipeline([("scaler", StandardScaler()), ("model", SVC())]),
        {"model__kernel": ["linear", "rbf"], "model__C": [0.1, 1, 10], "model__gamma": ["scale", "auto"]}
    ),
    "Decision Tree": (
        Pipeline([("model", DecisionTreeClassifier(random_state=2))]),
        {"model__criterion": ["gini", "entropy"], "model__max_depth": [2, 4, 6, None], "model__min_samples_split": [2, 5]}
    ),
    "KNN": (
        Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
        {"model__n_neighbors": [3, 5, 7, 9], "model__p": [1, 2]}
    )
}
results = []
fitted_models = {}
for name, (pipeline, params) in models.items():
    search = GridSearchCV(pipeline, params, cv=cv, scoring="accuracy", n_jobs=-1, error_score="raise")
    search.fit(X_train, y_train)
    pred = search.predict(X_test)
    fitted_models[name] = search.best_estimator_
    results.append({
        "Model": name,
        "Best CV Accuracy": search.best_score_,
        "Test Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "Best Parameters": str(search.best_params_)
    })
results_df = pd.DataFrame(results).sort_values("Test Accuracy", ascending=False)
display(results_df)
results_df.to_csv("model_comparison_results.csv", index=False)

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)
print("Best model by test accuracy (descriptive; test set is small):", best_name)
print(classification_report(y_test, best_pred, zero_division=0))
cm = confusion_matrix(y_test, best_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Failure (0)", "Success (1)"])
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig("best_model_confusion_matrix.png", dpi=160)
plt.show()
print("Confusion matrix [rows=actual, columns=predicted]:\n", cm)

## Interpretation template
After running, record the best model and its measured metrics from `model_comparison_results.csv`. Discuss precision, recall and F1 in addition to accuracy. A small test set can make metrics unstable, and the model with the highest test score is not necessarily the best generalizing model. Do not copy performance figures from an earlier draft unless these outputs reproduce them.